# VLLM
## What is vLLM?
vLLM is an open-source library for running large language models (LLMs) quickly and efficiently. This means it takes a trained model and makes it available to respond to requests. vLLM is one of three popular open-source LLM inference engines, the other two being SGLang and TensorRT-LLM. All are built on top of CUDA. If you want to learn more about how to choose an LLM inference engine for your use case, check out our LLM Engineer’s Almanac.

### What does the “v” in vLLM stand for?
Originally, the “v” stood for virtual, reflecting the project’s goal to make large models lightweight and deployable. Today, vLLM is increasingly associated with Vision + Language models — systems that can interpret both images and text (e.g., captioning tools or visual question-answering systems).


## Create conda env vLLM and install
- Create a conda environment
- Install Vllm with Cuda enable on SuperPOD

```
$ module load conda gcc/13 cuda/12
$ conda create --prefix=~/vllm python=3.12
$ conda activate ~/vllm
$ uv pip install vllm --torch-backend=auto
```

## Deploy vLLM
### Download vLLM model

- vLLM downloads and uses Huggingface model, for example, one can download models from HuggingFace:
    - https://huggingface.co/openai/gpt-oss-20b
    - https://huggingface.co/google/gemma-4-E2B

```
$ vllm serve openai/gpt-oss-20b
$ vllm serve google/gemma-4-E2B
```

### List downloaded models:
- By default, vLLM (and HuggingFace) downloads model to user's home directory, under hidden **.cache** folder

```
$ ls ~/.cache/huggingface/hub | grep models--
```

- Due to limited personal home folder's storage in SuperPOD, we encourage user to download models to project storage allocated by Cold Front HPC management system by setting **HF_HOME**:

```
$ export HF_HOME=/project_storage
```
- Redownload again and check the downloaded models:

```
$ ls $HF_HOME/hub
```

### Serving vLLM:

- In order to use vLLM, you need to serve this model on local GPU node on SuperPOD.

```
$ vllm serve openai/gpt-oss-20b
```

- By default the vLLM server model is on http://0.0.0.0:8000
- However, due to the shared GPU node nature of SuperPOD, user should choose different port to avoid potential conflict.
- You can change the default port using:

```
$ vllm serve openai/gpt-oss-20b --port 1234
```

## Run model's chat
- Once served, you can run the chat in python code or in Jupyter Notebook:


In [2]:
from openai import OpenAI
question = "write a poem about SMU in Dallas"

client = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="EMPTY",  # vLLM usually accepts any placeholder
)

resp = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": question}],
    temperature=0,
)

print(resp.choices[0].message.content)

The user just wants a poem. We can produce a poem. Let's write a poem that references SMU, Dallas, the campus, the Mustangs, the city, the culture, the architecture, the student life, the history, the "University Park" area, the "SMU" "Cameron" etc. We can write in free verse or a structured form. Let's do a free verse with some imagery. We'll mention the "Miller" building, the "Cameron" building, the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cam

### Serving multiple vLLMs
- Sometime user want to run 2 or more LLMs, just like in a chatbot where 1 LLM is for regular answer, another is for image querying and another for code.
- We can host 2 different models on 2 different port numbers with pre-allocated gpu-memory-utilization for each model size to avoid cuda overhead

```
$ vllm serve openai/gpt-oss-20b --port=1234 --gpu-memory-utilization 0.30 & 
$ vllm serve google/gemma-4-E2B-it --port=4321 --gpu-memory-utilization 0.30 & 
```

In [10]:
from openai import OpenAI
question = "write a poem about SMU in Dallas"

client1 = OpenAI(
    base_url="http://localhost:1234/v1",
    api_key="EMPTY",  # vLLM usually accepts any placeholder
)

resp1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": question}],
    temperature=0,
)

print(resp1.choices[0].message.content)


The user just wants a poem. We can produce a poem. Let's write a poem that references SMU, Dallas, the campus, the Mustangs, the city, the culture, the architecture, the student life, the history, the "University Park" area, the "SMU" "Cameron" etc. We can write in free verse or a structured form. Let's do a free verse with some imagery. We'll mention the "Miller" building, the "Cameron" building, the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cameron" etc. We'll mention the "SMU" "Cam

In [12]:

client2 = OpenAI(
    base_url="http://localhost:4321/v1",
    api_key="EMPTY",  # vLLM usually accepts any placeholder
)

resp2 = client2.chat.completions.create(
    model="google/gemma-4-E2B-it",
    messages=[{"role": "user", "content": question}],
    temperature=0,
)

print(resp2.choices[0].message.content)

## The Spirit of SMU in Dallas

Where the DFW winds begin to blow,
And the city's vibrant currents flow,
Stands a beacon, strong and bright,
A place of learning, bathed in light.
SMU, in Dallas' embrace,
A legacy etched in time and space.

From campus grounds where knowledge gleams,
Awakening ambitious dreams,
The halls echo with eager sound,
Where futures are profoundly found.
A tapestry of minds entwined,
With passion for the world defined.

The spirit here is bold and keen,
A dynamic, energetic scene.
In classrooms bright, the theories bloom,
Dispelling shadows, conquering gloom.
From engineering's sharp design,
To arts and sciences so divine,
A diverse chorus starts to rise,
Reflected in each hopeful eyes.

The football roar, a thunderous beat,
A shared allegiance, bittersweet,
A sense of pride that runs so deep,
The promises the students keep.
Beneath the Texas sky so wide,
With innovation as their guide.

The friendships forged in shared pursuit,
Bearing the harvest of the root
O

### To kill vllm instance:

```
$ pkill -f "vllm serve"
```